# Checking output files

In [1]:
from datetime import datetime, timedelta
import calendar
import collections
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
import sys

import glob
import pandas as pd
import xarray as xr
from IPython.display import display, HTML

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


my_dir = "/g/data/eg3/spr548/projects/"
sys.path.append(os.path.join(my_dir, "nesp_bff")+os.sep)
from nathers import location_details

# import utils
# from utils import locations, model_dict, vars_1hr, vars_day

In [13]:
step = "step2"
show_only_if_flagged = True

#==========================================
root_dir = "/g/data/eg3/nesp_bff/"
# location = "Sydney"
# scenario = "ssp370"
# model = "CESM2"
# time_period = "2041-2060"
vars_to_summarise_step2 = ["tas","huss","sfcWind","psl","uas","vas","clt","rsds","rsdsdir","rsdsdif"]
vars_to_summarise_step3 = ["tas","twbt","huss","psl","wind_speed","wind_dir","total_cloud_cover","rsds","rsdsdir","rsdsdif"]
locations = ["Darwin","Cairns","Brisbane","Longreach","Mildura","Adelaide","Perth","Sydney","Melbourne","Canberra","Hobart"]
vars_maybe_drop = ["time_offset", "round_method", "crs", "lat", "lon"]

input_dir = f"{root_dir}step3_calc_missing_vars/" if step == "step3" else f"{root_dir}step2_qdc_scaling/BARPA-R/"
vars_to_summarise = vars_to_summarise_step3 if step == "step3" else vars_to_summarise_step2

In [12]:
files = glob.glob(f"{input_dir}*.nc")
len(files)

462

In [14]:
qc_rows = []
errors = []

for file in sorted(files):
    base = file.split("/")[-1]
    parts = base.split("_")

    loc = parts[0] if len(parts) > 0 else None
    model_ = parts[2] if len(parts) > 2 else None
    ssp = parts[3] if len(parts) > 3 else None
    time_period_ = parts[9] if len(parts) > 9 else None

    header = f"{loc}: {model_}, {ssp}, {time_period_}"
    print(f"==================== {header} ====================")

    try:
        with xr.open_dataset(file) as da:
            df = (
                da.drop_vars([v for v in vars_maybe_drop if v in da.variables])
                  [vars_to_summarise]
                  .to_dataframe()
            )

        desc = df.describe()

        flagged = df.isna().any().any() or (df.nunique(dropna=True) <= 1).any()

        if (not show_only_if_flagged) or flagged:
            print("⚠️ Flagged" if flagged else "OK")
            display(HTML('<div style="overflow-x:auto; max-width:100%;">'))
            display(desc)
            display(HTML("</div>"))

        qc_rows.append({
            "file": base,
            "loc": loc,
            "model": model_,
            "ssp": ssp,
            "time_period": time_period_,
            "n_min": int(desc.loc["count"].min()),
            "any_nan": bool(df.isna().any().any()),
            "any_const": bool((df.nunique(dropna=True) <= 1).any()),
            "rsds_min": float(desc.loc["min", "rsds"]) if "rsds" in desc.columns else None,
            "rsds_max": float(desc.loc["max", "rsds"]) if "rsds" in desc.columns else None,
        })

    except Exception as e:
        errors.append({
            "file": base,
            "loc": loc,
            "model": model_,
            "ssp": ssp,
            "time_period": time_period_,
            "error": repr(e),
        })

qc_df = pd.DataFrame(qc_rows)
err_df = pd.DataFrame(errors)

print("\n=== QC summary (all files) ===")
display(qc_df)

if not err_df.empty:
    print("\n=== Errors ===")
    display(err_df)

==================== Adelaide: ACCESS-CM2, ssp126, 2021-2040 ====================
==================== Adelaide: ACCESS-CM2, ssp126, 2041-2060 ====================
⚠️ Flagged


,tas,huss,sfcWind,psl,uas,vas,clt,rsds,rsdsdir,rsdsdif
count,175320.000000,175320.000000,175320.000000,175320.000000,175272.000000,175272.000000,175272.000000,175272.000000,175319.000000,175320.000000
mean,291.385535,0.007278,3.017569,101227.236169,0.329787,0.842004,44.470101,215.088843,218.944491,64.831755
std,6.816063,0.002185,1.932564,728.265088,3.150072,3.481682,38.724042,303.768071,345.802558,100.062575
min,274.657444,0.000393,0.000000,97528.480864,-12.920898,-12.734375,0.000000,0.000000,0.000000,0.000000
25%,286.556318,0.005794,1.431127,100749.384648,-1.917969,-1.500000,0.666016,0.000000,0.000000,0.000000
50%,290.131241,0.007049,2.935577,101208.408725,-0.251953,1.119141,40.390625,7.476562,0.000000,0.000000
75%,295.329311,0.008450,4.358450,101695.640976,2.562500,3.388672,84.832031,387.007812,382.762568,92.501919
max,319.746069,0.021823,13.799576,103618.296863,14.207031,11.372070,100.000000,1125.656250,1296.002304,723.055769


==================== Adelaide: ACCESS-CM2, ssp126, 2061-2080 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2021-2040 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2041-2060 ====================
==================== Adelaide: ACCESS-CM2, ssp370, 2061-2080 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2021-2040 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2041-2060 ====================
==================== Adelaide: ACCESS-ESM1-5, ssp126, 2061-2080 ====================
⚠️ Flagged


,tas,huss,sfcWind,psl,uas,vas,clt,rsds,rsdsdir,rsdsdif
count,175320.000000,175320.000000,175320.000000,175320.000000,175224.000000,175224.000000,175224.000000,175224.000000,175317.000000,175320.000000
mean,291.121911,0.007048,3.095351,101196.310087,0.502681,0.813736,41.139427,220.309468,219.872807,64.126066
std,6.701048,0.001945,1.951241,715.838373,3.159871,3.456609,38.300179,309.463621,345.418740,97.842820
min,274.796778,0.000315,0.000000,97727.649069,-12.100586,-12.073242,0.000000,0.000000,0.000000,0.000000
25%,286.372353,0.005740,1.512935,100711.925061,-1.777344,-1.521484,0.000000,0.000000,0.000000,0.000000
50%,289.799872,0.006849,3.056373,101195.045345,-0.055664,1.051758,33.410156,7.910156,0.000000,0.000000
75%,294.815562,0.008147,4.479700,101690.320890,2.750977,3.331055,79.807129,399.595703,386.964211,92.651711
max,319.976718,0.019094,14.285907,103256.669251,12.871094,11.301758,100.000000,1129.531250,1296.081298,721.370234


==================== Adelaide: ACCESS-ESM1-5, ssp370, 2021-2040 ====================
⚠️ Flagged


...

,tas,huss,sfcWind,psl,uas,vas,clt,rsds,rsdsdir,rsdsdif
count,175320.000000,175320.000000,175320.000000,175320.000000,175272.000000,175272.000000,175272.000000,175272.000000,175319.000000,175320.000000
mean,292.368340,0.009298,5.372589,101659.959392,0.029983,0.133704,41.611471,209.830443,204.429530,68.790136
std,5.180563,0.003361,2.918417,721.173179,2.287112,2.572600,39.810640,290.210463,329.196021,105.281505
min,276.186946,0.001099,0.000000,98109.328687,-7.608398,-8.153320,0.000000,0.000000,0.000000,0.000000
25%,288.738618,0.006607,3.156332,101181.800171,-1.539062,-1.861328,0.000000,0.000000,0.000000,0.000000
50%,292.505073,0.009108,4.831606,101699.567905,-0.024414,-0.038574,30.220703,7.968750,0.000000,0.000000
75%,295.848537,0.011797,7.276444,102171.940461,1.375977,1.799805,85.285645,399.907227,332.373375,98.215691
max,319.887292,0.019799,83.812563,103732.845938,10.930664,12.626953,100.000000,1122.343750,1292.421917,683.354490


==================== Sydney: ACCESS-ESM1-5, ssp370, 2041-2060 ====================
==================== Sydney: ACCESS-ESM1-5, ssp370, 2061-2080 ====================
==================== Sydney: CESM2, ssp126, 2021-2040 ====================
==================== Sydney: CESM2, ssp126, 2041-2060 ====================
⚠️ Flagged


,tas,huss,sfcWind,psl,uas,vas,clt,rsds,rsdsdir,rsdsdif
count,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,175200.000000,166440.000000,175200.000000,175194.000000
mean,293.058195,0.009762,5.349598,101721.497987,0.098819,0.456339,44.429320,206.404806,208.417499,68.738182
std,5.149735,0.003397,2.920050,718.111819,2.296954,2.661743,39.839572,285.784495,333.346739,105.548939
min,277.314095,0.001078,0.000000,98551.912038,-9.519531,-8.971680,0.000000,0.000000,0.000000,0.000000
25%,289.476439,0.007082,3.129902,101243.471782,-1.475586,-1.621094,0.074219,0.000000,0.000000,0.000000
50%,293.257164,0.009661,4.801595,101737.699451,0.067383,0.399414,37.580078,7.666016,0.000000,0.000000
75%,296.455516,0.012379,7.288040,102217.694804,1.409180,2.210938,88.285156,391.297852,348.761426,96.453629
max,320.786767,0.020519,91.625561,104111.995900,11.110352,11.063477,100.000000,1118.656250,1298.289686,711.197684


==================== Sydney: CESM2, ssp126, 2061-2080 ====================
==================== Sydney: CESM2, ssp370, 2021-2040 ====================
==================== Sydney: CESM2, ssp370, 2041-2060 ====================
==================== Sydney: CESM2, ssp370, 2061-2080 ====================
==================== Sydney: CMCC-ESM2, ssp126, 2021-2040 ====================
==================== Sydney: CMCC-ESM2, ssp126, 2041-2060 ====================
==================== Sydney: CMCC-ESM2, ssp126, 2061-2080 ====================
==================== Sydney: CMCC-ESM2, ssp370, 2021-2040 ====================
==================== Sydney: CMCC-ESM2, ssp370, 2041-2060 ====================
==================== Sydney: CMCC-ESM2, ssp370, 2061-2080 ====================
==================== Sydney: EC-Earth3, ssp126, 2021-2040 ====================
==================== Sydney: EC-Earth3, ssp126, 2041-2060 ====================
==================== Sydney: EC-Earth3, ssp126, 2061-2080 ==========

,file,loc,model,ssp,time_period,n_min,any_nan,any_const,rsds_min,rsds_max
0,Adelaide_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp126,2021-2040,175320,False,False,0.0,1133.582031
1,Adelaide_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp126,2041-2060,175272,True,False,0.0,1125.656250
2,Adelaide_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp126,2061-2080,175320,False,False,0.0,1127.695312
3,Adelaide_AUS-15_ACCESS-CM2_ssp370_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp370,2021-2040,175320,False,False,0.0,1123.550781
4,Adelaide_AUS-15_ACCESS-CM2_ssp370_r4i1p1f1_BOM...,Adelaide,ACCESS-CM2,ssp370,2041-2060,175320,False,False,0.0,1125.070312
...,...,...,...,...,...,...,...,...,...,...
457,Sydney_AUS-15_NorESM2-MM_ssp126_r1i1p1f1_BOM_B...,Sydney,NorESM2-MM,ssp126,2041-2060,175200,False,False,0.0,1127.769531
458,Sydney_AUS-15_NorESM2-MM_ssp126_r1i1p1f1_BOM_B...,Sydney,NorESM2-MM,ssp126,2061-2080,175200,False,False,0.0,1124.964844
459,Sydney_AUS-15_NorESM2-MM_ssp370_r1i1p1f1_BOM_B...,Sydney,NorESM2-MM,ssp370,2021-2040,175200,False,False,0.0,1132.242188
460,Sydney_AUS-15_NorESM2-MM_ssp370_r1i1p1f1_BOM_B...,Sydney,NorESM2-MM,ssp370,2041-2060,175200,False,False,0.0,1125.160156


In [ ]:
# for file in files:
#     base = file.split('/')[-1]
#     loc = base.split('_')[0]
#     model = base.split('_')[2]
#     ssp = base.split('_')[3]
#     time_period = base.split('_')[9]
#     print(f"========================== {loc}: {model}, {ssp}, {time_period} ==========================")
#     da = xr.open_dataset(file)
#     print(da.drop_vars([v for v in ["time_offset","round_method",
#                                     "crs", "lat", "lon"] if v in da.variables])[vars_to_summarise].to_dataframe().describe())